# 04 YOLO11 b-case fallback crop pose 테스트

full-frame YOLO11 pose와 custom YOLO11 detector를 같은 frame에 적용한 뒤, custom detector만 잡은 b 케이스를 crop batch로 다시 pose 추론합니다. crop들은 한 장으로 merge하지 않고 list batch로 model에 전달합니다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행하거나 REPO_ROOT를 직접 지정하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("REPO_ROOT:", REPO_ROOT)

from labelstudio_bbox_tools.pose_inference.fallback_yolo11 import run_yolo11_pose_fallback_crop_test

## 설정

처음에는 `RUN_PREVIEW=False`, `RUN_INFERENCE=False`, `DRY_RUN=True`라서 아무 것도 실행하지 않습니다. 실제 테스트 전 `INPUT_PATH`, `OUT_DIR`, `DETECTOR_WEIGHTS`, class 설정을 채우세요.

In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

INPUT_PATH = Path("/path/to/video_or_video_folder")
OUT_DIR = Path("/path/to/pose_fallback_outputs")

# custom detection weight: full-frame에서 사람/작업자 후보를 찾는 모델입니다.
DETECTOR_WEIGHTS = Path("/path/to/custom_detector.pt")

# 실제 실행 전 custom detector의 class YAML을 넣는 것을 권장합니다.
# preview만 볼 때는 아래 MANUAL_CLASSES 기본값으로도 동작합니다.
DETECTOR_CLASS_YAML = None
DETECTOR_MANUAL_CLASSES = ["person"]
TARGET_DETECTION_CLASSES = []  # 예: ["person", "worker", "fall"]. 비우면 detector 결과 전체를 pose 후보로 사용합니다.

# full-frame pose와 fallback crop pose 모두 같은 official pose weight를 사용합니다.
POSE_WEIGHTS = "yolo11x-pose.pt"
POSE_CLASS_YAML = None
POSE_MANUAL_CLASSES = []

DEVICE = "cuda:0"
DETECTOR_IMGSZ = 640
POSE_IMGSZ = 640
DETECTOR_CONF = 0.25
DETECTOR_IOU = 0.50
POSE_CONF = 0.25
POSE_IOU = 0.45
KEYPOINT_CONF = 0.20

# full-frame pose bbox와 custom detector bbox가 이 IoU 이상이면 c 케이스로 보고, 아니면 b/a로 분리합니다.
MATCH_IOU = 0.30

# b 케이스 fallback crop 설정입니다. crop들은 한 장으로 붙이지 않고 list batch로 pose model에 넣습니다.
CROP_PADDING_RATIO = 0.15
MIN_CROP_SIZE = 32
FALLBACK_BATCH_SIZE = 8
MAX_FALLBACK_CROPS_PER_FRAME = 4
MAX_POSE_PER_CROP = 1
FINAL_NMS_IOU = 0.50

RECURSIVE = False
MAX_VIDEOS = 1
FRAME_STRIDE = 1
START_SECONDS = None
END_SECONDS = None
MAX_FRAMES = 300

FONT_PATH = None
FONT_SIZE = 20
LINE_WIDTH = 3
KEYPOINT_RADIUS = 4
RUN_NAME = None
OVERWRITE = False

DRAW_BBOX = True
DRAW_SKELETON = True
DRAW_KEYPOINTS = True

# ===== 안전 플래그 =====
RUN_PREVIEW = False
RUN_INFERENCE = False
DRY_RUN = True

## 1. Preview

영상 목록과 metadata만 확인합니다. 모델을 load하지 않습니다.

In [ ]:
if RUN_PREVIEW:
    preview = run_yolo11_pose_fallback_crop_test(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        detector_weights=DETECTOR_WEIGHTS,
        detector_class_yaml=DETECTOR_CLASS_YAML,
        detector_manual_classes=DETECTOR_MANUAL_CLASSES,
        pose_weights=POSE_WEIGHTS,
        pose_class_yaml=POSE_CLASS_YAML,
        pose_manual_classes=POSE_MANUAL_CLASSES,
        target_detection_classes=TARGET_DETECTION_CLASSES,
        device=DEVICE,
        detector_imgsz=DETECTOR_IMGSZ,
        pose_imgsz=POSE_IMGSZ,
        detector_conf=DETECTOR_CONF,
        detector_iou=DETECTOR_IOU,
        pose_conf=POSE_CONF,
        pose_iou=POSE_IOU,
        keypoint_conf=KEYPOINT_CONF,
        match_iou=MATCH_IOU,
        crop_padding_ratio=CROP_PADDING_RATIO,
        min_crop_size=MIN_CROP_SIZE,
        fallback_batch_size=FALLBACK_BATCH_SIZE,
        max_fallback_crops_per_frame=MAX_FALLBACK_CROPS_PER_FRAME,
        max_pose_per_crop=MAX_POSE_PER_CROP,
        final_nms_iou=FINAL_NMS_IOU,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        keypoint_radius=KEYPOINT_RADIUS,
        draw_bbox=DRAW_BBOX,
        draw_skeleton=DRAW_SKELETON,
        draw_keypoints=DRAW_KEYPOINTS,
        overwrite=OVERWRITE,
        dry_run=True,
    )
    preview.as_dict()
else:
    print("RUN_PREVIEW=False 이므로 preview를 건너뜁니다.")

## 2. Fallback Crop Inference

실제 실행 전 `RUN_INFERENCE=True`, `DRY_RUN=False`로 바꿉니다. 결과에는 `predictions.jsonl`과 `fallback_cases_summary.json`이 생성됩니다.

In [ ]:
if RUN_INFERENCE:
    result = run_yolo11_pose_fallback_crop_test(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        detector_weights=DETECTOR_WEIGHTS,
        detector_class_yaml=DETECTOR_CLASS_YAML,
        detector_manual_classes=DETECTOR_MANUAL_CLASSES,
        pose_weights=POSE_WEIGHTS,
        pose_class_yaml=POSE_CLASS_YAML,
        pose_manual_classes=POSE_MANUAL_CLASSES,
        target_detection_classes=TARGET_DETECTION_CLASSES,
        device=DEVICE,
        detector_imgsz=DETECTOR_IMGSZ,
        pose_imgsz=POSE_IMGSZ,
        detector_conf=DETECTOR_CONF,
        detector_iou=DETECTOR_IOU,
        pose_conf=POSE_CONF,
        pose_iou=POSE_IOU,
        keypoint_conf=KEYPOINT_CONF,
        match_iou=MATCH_IOU,
        crop_padding_ratio=CROP_PADDING_RATIO,
        min_crop_size=MIN_CROP_SIZE,
        fallback_batch_size=FALLBACK_BATCH_SIZE,
        max_fallback_crops_per_frame=MAX_FALLBACK_CROPS_PER_FRAME,
        max_pose_per_crop=MAX_POSE_PER_CROP,
        final_nms_iou=FINAL_NMS_IOU,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        keypoint_radius=KEYPOINT_RADIUS,
        draw_bbox=DRAW_BBOX,
        draw_skeleton=DRAW_SKELETON,
        draw_keypoints=DRAW_KEYPOINTS,
        overwrite=OVERWRITE,
        dry_run=DRY_RUN,
    )
    result.as_dict()
else:
    print("RUN_INFERENCE=False 이므로 실제 inference를 건너뜁니다.")